In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

In [ ]:
import numpy as np
from collections import deque
import matplotlib.pyplot as plt

In [ ]:
import gymnasium as gym
import flappy_bird_gymnasium

In [ ]:
from gymnasium.wrappers import (
    GrayscaleObservation,
    ResizeObservation,
    FrameStackObservation,
)

In [ ]:
# Custom wrapper to return RGB screen frames as observations
class PixelWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        # FlappyBird render output shape is (512, 288, 3)
        self.observation_space = gym.spaces.Box(
            low=0, high=255, shape=(512, 288, 3), dtype=np.uint8
        )

    def observation(self, observation):
        return self.env.render()

**Step 1: Update Environment Setup**  
Replace with `gym.vector.AsyncVectorEnv` using a helper generator function:

In [ ]:
NUM_ENVS = 6  # Number of parallel environments (adjust based on CPU cores)

In [ ]:
def make_env():
    def _thunk():
        # Import inside worker process so Gymnasium registers FlappyBird-v0 in sub-processes
        import flappy_bird_gymnasium
        
        env = gym.make("FlappyBird-v0", render_mode="rgb_array")
        env = PixelWrapper(env)
        env = GrayscaleObservation(env)
        env = ResizeObservation(env, (84, 84))
        env = FrameStackObservation(env, stack_size=4)
        return env
    return _thunk

# Vectorized parallel environments for training
envs = gym.vector.AsyncVectorEnv([make_env() for _ in range(NUM_ENVS)])

# Single environment reserved for evaluation / video recording
eval_env = make_env()()

states, _ = envs.reset()
print("Vectorized Observation Shape:", states.shape)  # Output: (6, 4, 84, 84)

In [ ]:
# Extract space information from the single evaluation environment
s_size = eval_env.observation_space.shape
a_size = eval_env.action_space.n

print(f"State space shape: {s_size}")  # Output: (4, 84, 84)
print(f"Action space size: {a_size}")  # Output: 2

print("\n_____OBSERVATION SPACE_____")
print("The State Space is:", s_size)
print("Sample observation shape:", eval_env.observation_space.sample().shape)

print("\n_____ACTION SPACE_____")
print("The Action Space is:", a_size)
print("Action Space Sample:", eval_env.action_space.sample())

### Observation Space (Pixel-Based)

Instead of the 12-feature vector, the agent receives stacked raw screen frames processed through a Gym wrapper pipeline.

* **Observation Shape:** `(4, 84, 84)` — `(Frame Stack, Height, Width)`
* **Data Type:** Grayscale pixel values ranging from `0` to `255` (`uint8`)

**Preprocessing Pipeline**
* **Frame Extraction:** Captures raw screen renders `(512, 288, 3)` directly from the environment using `render_mode="rgb_array"`.
* **Grayscale Conversion:** Converts RGB images down to a single channel (`GrayscaleObservation`).
* **Spatial Downsampling:** Resizes resolution down to $84 \times 84$ pixels to minimize network parameters (`ResizeObservation`).
* **Frame Stacking:** Stacks `4` consecutive frames (`FrameStackObservation`) to enable the CNN to perceive velocity, falling acceleration, and pipe movement speed.

---

**Step 2: Update `CNNPolicy` Act Method**
Because `envs.step()` now expects and returns batch arrays of shape `(NUM_ENVS, ...)`, update `act()` to handle batch tensors directly without calling `.item()`:

---

Wrap Inference with `@torch.no_grad()` inside `act`  
so states_tensor and action sampling don't build unneeded computational graphs during the forward step:

---

In [ ]:
class CNNPolicy(nn.Module):
    def __init__(self, in_channels=4, a_size=2):
        super(CNNPolicy, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten()
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, a_size)
        )

    def forward(self, x):
        x = self.conv(x)
        return F.softmax(self.fc(x), dim=1)

    @torch.no_grad()
    def act(self, states):
        states_tensor = torch.from_numpy(np.array(states)).float().to(device) / 255.0
        probs = self.forward(states_tensor)
        m = Categorical(probs)
        actions = m.sample()
        return actions.cpu().numpy()

### Define the hyperparameters

---

Update Cell to pull space metadata directly from `eval_env`:

---

In [ ]:
# Define environment metadata variables
env_id = "FlappyBird-v0"
s_size = eval_env.observation_space.shape
a_size = eval_env.action_space.n

In [ ]:
hyperparameters = {
    "h_size": 64,
    "n_training_episodes": 200000,
    "n_evaluation_episodes": 10,
    "max_t": 10000,
    "gamma": 0.99,
    "lr": 1e-3,
    "env_id": env_id,
    "state_space": s_size,
    "action_space": a_size,
}

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
print(device)

In [ ]:
# Create policy and place it to the device
policy = CNNPolicy(in_channels=4, a_size=a_size).to(device)

optimizer = optim.Adam(policy.parameters(), lr=hyperparameters["lr"])

---
### Step 3: Vectorized `reinforce_parallel` Training Loop

Replaced the standard loop with a vectorized REINFORCE loop that steps all parallel environments together. To prevent memory leaks, OOM crashes, and local minimum traps, the following architectural changes were implemented:

**1. CPU Trajectory Buffering (`uint8` arrays)**
*   **What:** Rollout frames are stored in system RAM as lightweight `uint8` NumPy arrays rather than PyTorch GPU tensors.
*   **Why:** Entirely fixes the CUDA Out of Memory (OOM) error. It prevents PyTorch from building massive computational graphs that cause the "staircase" of VRAM usage during the episode loop.

**2. Mini-Batched GPU Backpropagation (`chunk_size=256`)**
*   **What:** Gradients are calculated and backpropagated in small chunks of 256 frames at a time.
*   **Why:** Keeps GPU VRAM usage permanently flat (under ~200 MB), allowing for infinitely long episodes without crashing.

**3. Safe Return Standardization (`ret_std > 1e-5`)**
*   **What:** Added a variance guard to ensure we only divide by the standard deviation if there is actual variance in the rewards.
*   **Why:** Fixes the stuck average score of 0.80. When all agents died at the exact same time, the variance was zero. Standardizing a zero-variance array previously wiped out all gradients (perfect gradient cancellation), preventing the network from learning.

**4. Entropy Regularization (`- entropy_coef * entropy`)**
*   **What:** The policy's entropy (randomness) is subtracted from the loss function, penalizing the agent for being too deterministic/predictable.
*   **Why:** Vanilla REINFORCE easily collapses into a state where it just spams "Action 0" (do nothing). This forces the agent to keep trying "Action 1" (flap) so it can eventually discover how to pass the pipes.

**5. Scaled Loss (Averaging over batch)**
*   **What:** The loss is divided by `total_samples` to get the true batch mean.
*   **Why:** Prevents gradients from exploding as the agent gets better and survives for thousands of frames per episode.
---

In [ ]:
def reinforce_parallel(envs, policy, optimizer, n_training_episodes, max_t, gamma, print_every=600, chunk_size=256, entropy_coef=0.1):
    scores_deque = deque(maxlen=100)
    scores = []
    num_envs = envs.num_envs
    total_episodes_completed = 0
    last_printed_episodes = 0
    
    # 1. Initialize states ONCE before the loop (envs will auto-reset independently upon death)
    states, _ = envs.reset()
    
    # Track rewards continuously across chunks to get accurate episode scores
    current_ep_rewards = np.zeros(num_envs, dtype=np.float32)
    
    while total_episodes_completed < n_training_episodes:
        batch_states = []
        batch_actions = []
        rewards = []
        dones = []
        
        # 2. Collect rollout trajectory in fast, fixed chunks
        ROLLOUT_STEPS = 256 
        
        for t in range(ROLLOUT_STEPS):
            actions = policy.act(states)
            next_states, step_rewards, terminated, truncated, _ = envs.step(actions)

            # Reward survival: add +0.1 per frame to give the gradient continuous feedback
            step_rewards = step_rewards + 0.1
            
            batch_states.append(np.array(states, dtype=np.uint8))
            batch_actions.append(actions)
            rewards.append(step_rewards)
            
            done = terminated | truncated
            dones.append(done) 
            
            # Accurately track full episode scores despite fixed rollout lengths
            current_ep_rewards += step_rewards
            for i in range(num_envs):
                if done[i]:
                    scores_deque.append(current_ep_rewards[i])
                    scores.append(current_ep_rewards[i])
                    current_ep_rewards[i] = 0
                    total_episodes_completed += 1
            
            states = next_states

        # 3. Compute discounted returns across auto-resets
        discounted_returns = []
        g = np.zeros(num_envs, dtype=np.float32)
        
        # Iterate backwards through the rewards and the death flags
        for r, d in zip(reversed(rewards), reversed(dones)):
            # (1.0 - d) acts as a mask. Multiplies future returns by 0 if the bird died, 
            # preventing rewards from the new game bleeding into the old game.
            g = r + gamma * g * (1.0 - d) 
            discounted_returns.insert(0, g.copy())
            
        discounted_returns = np.array(discounted_returns, dtype=np.float32)
        
        # Safe return standardization
        ret_std = discounted_returns.std()
        if ret_std > 1e-5:
            discounted_returns = (discounted_returns - discounted_returns.mean()) / (ret_std + 1e-8)
        else:
            discounted_returns = discounted_returns - discounted_returns.mean()
        
        flat_states = np.concatenate(batch_states, axis=0)     
        flat_actions = np.concatenate(batch_actions, axis=0)   
        flat_returns = discounted_returns.reshape(-1)          
        
        total_samples = len(flat_actions)
        
        # 4. Mini-batched GPU backpropagation
        optimizer.zero_grad()
        for start_idx in range(0, total_samples, chunk_size):
            end_idx = min(start_idx + chunk_size, total_samples)
            
            b_states = torch.from_numpy(flat_states[start_idx:end_idx]).float().to(device) / 255.0
            b_actions = torch.from_numpy(flat_actions[start_idx:end_idx]).to(device)
            b_returns = torch.from_numpy(flat_returns[start_idx:end_idx]).to(device)
            
            probs = policy(b_states)
            m = Categorical(probs)
            log_probs = m.log_prob(b_actions)
            entropy = m.entropy()
            
            # SCALED LOSS: Divide by total_samples to get the true batch mean.
            policy_loss = -(torch.sum(log_probs * b_returns) / total_samples)
            entropy_loss = -entropy_coef * (torch.sum(entropy) / total_samples)
            
            chunk_loss = policy_loss + entropy_loss
            chunk_loss.backward()  
            
            last_loss = chunk_loss.item()
            
        optimizer.step()
        
        # 5. Print progress periodically
        if total_episodes_completed - last_printed_episodes >= print_every:
            mean_score = np.mean(scores_deque) if len(scores_deque) > 0 else 0
            print(f"Episodes Completed: {total_episodes_completed}\tAverage Score (last 100): {mean_score:.2f}\tLoss: {last_loss:.4f}")
            torch.save(policy.state_dict(), "flappybird_cnn_reinforce.pth")
            last_printed_episodes = total_episodes_completed
            
    return scores

In [ ]:
# # Load saved weights
# policy.load_state_dict(torch.load("flappybird_cnn_reinforce.pth"))
# policy.to(device)

Training Execution Cell: Update the execution line from reinforce(...) to reinforce_parallel(...) and pass envs:

In [ ]:
scores = reinforce_parallel(
    envs,
    policy,
    optimizer,
    hyperparameters["n_training_episodes"],
    hyperparameters["max_t"],
    hyperparameters["gamma"],
    print_every=600  # Prints progress every 100 iterations across 6 workers
)

In [ ]:
# Save model weights to disk
torch.save(policy.state_dict(), "flappybird_cnn_reinforce.pth")
print("Model checkpoint saved successfully!")

In [ ]:
from gymnasium.wrappers import RecordVideo
from IPython.display import Video
import os
import glob
import torch
import numpy as np

# 1. Create a new evaluation environment wrapped with RecordVideo
video_folder = "flappybird-video"
os.makedirs(video_folder, exist_ok=True)

# Use your existing make_env logic or build a fresh single env with the exact wrapper pipeline
env_to_wrap = gym.make("FlappyBird-v0", render_mode="rgb_array")
env_to_wrap = PixelWrapper(env_to_wrap)
env_to_wrap = GrayscaleObservation(env_to_wrap)
env_to_wrap = ResizeObservation(env_to_wrap, (84, 84))
env_to_wrap = FrameStackObservation(env_to_wrap, stack_size=4)

video_env = RecordVideo(
    env_to_wrap, 
    video_folder=video_folder,
    episode_trigger=lambda e: True # Record every episode
)

# 2. Run a single evaluation episode to capture the video using your CNN policy
state, _ = video_env.reset()
done = False

with torch.no_grad():
    while not done:
        # Pass state through your policy's act method (which handles tensor conversion and batching)
        # Since policy.act expects a batched input, we wrap the single state in a list/array dimension
        action = policy.act(np.array([state]))[0]
        
        state, reward, terminated, truncated, _ = video_env.step(action)
        done = terminated or truncated

video_env.close()

# 3. Display the recorded video in Jupyter
video_files = glob.glob(f"{video_folder}/*.mp4")
if video_files:
    latest_video = max(video_files, key=os.path.getmtime)
    display(Video(latest_video, embed=True))
else:
    print("No video file found.")